# 09 — EMNIST Balanced: SDM and temperature calibration comparison

Use the **balanced** split with 47 zero-based classes, not the separate letters/digits variants. Apply the same upright transpose to training and evaluation images. Inspect the gallery before interpreting results.

This notebook reproduces **notebook 04's full reliability protocol** on EMNIST Balanced: one seed-42 ImageNet-pretrained ConvNeXt-Tiny model, two nested calibration sets, raw softmax, two temperature-scaled softmax baselines, and two frozen identity-adaptor SDM tracks. The classifier has **47 output classes**.

| Track | Full-run images/class | Total calibration images |
|---|---:|---:|
| `standard_25` | 25 | 1,175 |
| `enlarged_100` | 100 | 4,700 |

Each class also reserves 25 validation images. All remaining official training examples provide training/reference support; the official test split is evaluated after every fit is frozen. Unlike CIFAR-100, some datasets have unequal class sizes: report support and macro/micro metrics together.

Metrics match notebook 04: precision/recall (macro, micro, per-class, HR), fitted temperatures, NLL, Brier, ECE, error AUROC/AUPRC, risk–coverage/AURC, coverage at 95% selective accuracy, fixed-coverage accuracy, HR results, cohorts, confusion plots, and **macro recall versus coverage**. CSVs, PNGs, split manifests, logits, and embeddings are saved outside Git.

This is a new dataset experiment, not a claimed accuracy result. Keep rejected/weak runs; do not select checkpoints or datasets by final-test performance. The 95% target is an observed selective metric, not a distribution-shift guarantee.

Dataset API: [EMNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.EMNIST.html).


## Setup and experimental rules

Use Python 3.12 and a matching PyTorch/torchvision installation. From the repository root, activate your training environment, then run:

```bash
python -m pip install -r requirements-training.txt
python -m jupyterlab
```

For an NVIDIA installation, choose the CUDA wheel pair appropriate to your machine using the [official PyTorch installation selector](https://pytorch.org/get-started/locally/). CPU and Apple MPS are supported fallbacks, but full ConvNeXt runs are expensive. SVHN needs SciPy for its `.mat` files. Initial dataset/model downloads may be large; EMNIST downloads the archive containing all splits. `SDM_DOWNLOAD=0` uses prepared local data and weights only.

Choose **Restart Kernel and Run All Cells**. For a small pipeline check, start Jupyter with `SDM_RUN_MODE=smoke`; it uses 2 training, 1 validation, and nested 1/2 calibration examples per class, plus 2 test examples per class. The track names retain the nominal full-run 25/100 labels. Smoke results are incomplete and must not be interpreted. Smoke mode does not avoid the initial full dataset download.

1. Train with cross-entropy; select checkpoints using validation accuracy only.
2. Fit both SDM tracks, cohort maps, and softmax temperatures on calibration data only.
3. Freeze all fits before loading the official test split.
4. Evaluate all methods on identical test images; never tune from test outcomes.

`SDM_DATA_DIR` selects the shared download/cache root. `SDM_RUN_DIR` selects an output **root**; this notebook always appends dataset/model/seed/experiment/mode directories to prevent cross-dataset checkpoint collisions. Changing training settings requires a new output root or explicit `SDM_FORCE_RETRAIN=1`.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import time
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import InterpolationMode

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks" / "benchmark_datasets.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
from benchmark_datasets import DATASETS, dataset_transforms, load_dataset, stratified_splits, smoke_test_indices

DATASET_KEY = "emnist_balanced"
SPEC = DATASETS[DATASET_KEY]
NUM_CLASSES = SPEC.classes
DOWNLOAD = os.environ.get("SDM_DOWNLOAD", "1") == "1"

RUN_MODE = os.environ.get("SDM_RUN_MODE", "benchmark").strip().lower()
if RUN_MODE not in {"benchmark", "smoke"}:
    raise ValueError("SDM_RUN_MODE must be 'benchmark' or 'smoke'.")

SEED = 42
TARGET_ALPHA = 0.95
MODEL_NAME = "convnext_tiny_imagenet1k_v1"
EXPERIMENT_NAME = "calibration_25_vs_100"
DATA_DIR = Path(os.environ.get(
    "SDM_DATA_DIR", Path.home() / ".cache" / "sdm-vision-reliability"
)).expanduser()
RUN_ROOT = Path(os.environ.get("SDM_RUN_DIR", DATA_DIR / "runs")).expanduser()
RUN_DIR = RUN_ROOT / DATASET_KEY / MODEL_NAME / f"seed_{SEED}" / EXPERIMENT_NAME / RUN_MODE
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

SMOKE = RUN_MODE == "smoke"
IMAGE_SIZE = 224
BATCH_SIZE = 8 if SMOKE else int(os.environ.get("SDM_BATCH_SIZE", "128"))
EPOCHS = 1 if SMOKE else int(os.environ.get("SDM_EPOCHS", "20"))
if BATCH_SIZE < 1 or EPOCHS < 1:
    raise ValueError("Batch size and epochs must be positive.")
MAX_TRAIN_BATCHES = 2 if SMOKE else None
MAX_EXPORT_BATCHES = None  # Smoke uses explicit balanced subsets, not an ordered prefix.
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
NUM_WORKERS = 0 if SMOKE or platform.system() in {"Windows", "Darwin"} else min(4, os.cpu_count() or 1)
FORCE_RETRAIN = os.environ.get("SDM_FORCE_RETRAIN", "0") == "1"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"

CONFIG = {
    "dataset": DATASET_KEY,
    "number_of_classes": NUM_CLASSES,
    "source_notebook": "04_cifar100_sdm_calibration_comparison.ipynb",
    "protocol_issue": 28,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "target_alpha": TARGET_ALPHA,
    "model_name": MODEL_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "device": str(DEVICE),
    "calibration_images_per_class": [1, 2] if SMOKE else [25, 100],
    "validation_images_per_class": 1 if SMOKE else 25,
    "preprocessing_version": "dataset_specific_v1",
}
print(json.dumps(CONFIG, indent=2))
print(f"Outputs: {RUN_DIR}")

## Reproducibility: every stochastic source uses seed 42

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


seed_everything(SEED)

## Build disjoint training, validation, and nested calibration splits

Use a seed-42 permutation **within every true class**. Full runs reserve 25 validation and 100 enlarged calibration images/class; the standard set takes the first 25 of those calibration images. All remaining examples are training/reference support. This preserves natural class imbalance rather than silently truncating to a CIFAR-100-sized support.

Only training images receive stochastic augmentation. Evaluation is deterministic, retains the complete input field, and replicates grayscale channels for the pretrained RGB backbone. Handwriting and digits are never flipped. The class-labelled gallery lets you verify label mapping and EMNIST orientation before training.


In [ ]:
train_transform, eval_transform = dataset_transforms(SPEC, IMAGE_SIZE)
train_aug_base = load_dataset(SPEC, DATA_DIR / "datasets", train=True, transform=train_transform, download=DOWNLOAD)
train_eval_base = load_dataset(SPEC, DATA_DIR / "datasets", train=True, transform=eval_transform, download=False)
CLASS_NAMES = list(train_aug_base.classes)
targets = train_aug_base.targets
splits = stratified_splits(targets, NUM_CLASSES, seed=SEED, smoke=SMOKE)
train_indices = splits["train"]
validation_indices = splits["validation"]
calibration_25_indices = splits["standard"]
calibration_100_indices = splits["enlarged"]

assert set(calibration_25_indices) <= set(calibration_100_indices)
assert not set(train_indices) & set(validation_indices)
assert not set(train_indices) & set(calibration_100_indices)
assert not set(validation_indices) & set(calibration_100_indices)
assert len(set(train_indices) | set(validation_indices) | set(calibration_100_indices)) == (
    len(train_indices) + len(validation_indices) + len(calibration_100_indices)
)
if not SMOKE:
    assert len(train_indices) + len(validation_indices) + len(calibration_100_indices) == len(targets)
np.savez_compressed(RUN_DIR / "split_indices.npz", **splits)

loader_options = {
    "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda", "persistent_workers": NUM_WORKERS > 0,
}
train_loader = DataLoader(
    Subset(train_aug_base, train_indices), shuffle=True,
    generator=torch.Generator().manual_seed(SEED), **loader_options
)
reference_loader = DataLoader(Subset(train_eval_base, train_indices), shuffle=False, **loader_options)
validation_loader = DataLoader(Subset(train_eval_base, validation_indices), shuffle=False, **loader_options)
calibration_loader = DataLoader(Subset(train_eval_base, calibration_100_indices), shuffle=False, **loader_options)

split_table = pd.DataFrame({
    "class_name": CLASS_NAMES,
    **{name: np.bincount(targets[indices], minlength=NUM_CLASSES) for name, indices in splits.items()},
})
display(split_table)
print({"training/reference": len(train_indices), "validation": len(validation_indices),
       "standard calibration": len(calibration_25_indices), "enlarged calibration": len(calibration_100_indices),
       "official test loaded": False})

fig, axes = plt.subplots(2, 5, figsize=(12, 5), layout="constrained")
for class_id, axis in zip(np.linspace(0, NUM_CLASSES - 1, 10, dtype=int), axes.flat):
    image_index = np.flatnonzero(targets == class_id)[0]
    # Show the deterministic orientation/channel conversion used for evaluation.
    image, label = train_eval_base[image_index]
    image = image.permute(1, 2, 0).numpy() * np.asarray([.229, .224, .225]) + np.asarray([.485, .456, .406])
    axis.imshow(np.clip(image, 0, 1))
    axis.set_title(CLASS_NAMES[label])
    axis.axis("off")
fig.suptitle(f"{SPEC.title}: class-labelled evaluation inputs (ten sampled classes)")
fig.savefig(RUN_DIR / "dataset_gallery.png", dpi=160)
plt.show()


## Train the shared softmax model

ConvNeXt-Tiny starts from torchvision's fixed `IMAGENET1K_V1` weights. Cross-entropy applies log-softmax internally; no SDM value or test result affects gradient updates or checkpoint selection.

In [ ]:
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = convnext_tiny(weights=weights if DOWNLOAD else None)
if not DOWNLOAD:
    cached_weights = Path(torch.hub.get_dir()) / "checkpoints" / Path(weights.url).name
    if not cached_weights.is_file():
        raise FileNotFoundError(f"Pretrained weights unavailable offline: {cached_weights}")
    model.load_state_dict(torch.load(cached_weights, map_location="cpu", weights_only=True))
model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def classification_epoch(loader, *, training: bool, max_batches: int | None = None) -> dict:
    model.train(training)
    total_loss = total_examples = top1_correct = top5_correct = 0
    for batch_number, (inputs, labels) in enumerate(loader):
        if max_batches is not None and batch_number >= max_batches:
            break
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(inputs)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        count = labels.size(0)
        top5 = logits.topk(min(5, NUM_CLASSES), dim=1).indices
        total_loss += loss.item() * count
        total_examples += count
        top1_correct += (top5[:, 0] == labels).sum().item()
        top5_correct += (top5 == labels[:, None]).any(dim=1).sum().item()
    return {
        "loss": total_loss / total_examples,
        "top1_accuracy": top1_correct / total_examples,
        "top5_accuracy": top5_correct / total_examples,
        "examples": total_examples,
    }


checkpoint_path = RUN_DIR / "best_checkpoint.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False) if checkpoint_path.exists() and not FORCE_RETRAIN else None
compatibility_keys = (
    "dataset", "number_of_classes", "run_mode", "seed", "model_name", "image_size",
    "epochs", "batch_size", "learning_rate", "weight_decay", "label_smoothing",
    "calibration_images_per_class", "validation_images_per_class", "preprocessing_version",
)
if checkpoint:
    if any(checkpoint.get("config", {}).get(key) != CONFIG[key] for key in compatibility_keys):
        raise ValueError("Checkpoint configuration mismatch. Use a new SDM_RUN_DIR or SDM_FORCE_RETRAIN=1.")
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError("Checkpoint class mapping mismatch.")
    if not checkpoint.get("training_complete", False):
        print("Restarting incomplete training; complete checkpoints alone are reused.")
        checkpoint = None
history = checkpoint.get("history", []) if checkpoint else []
best_validation_accuracy = checkpoint.get("best_validation_accuracy", -1.0) if checkpoint else -1.0

if checkpoint and checkpoint.get("training_complete", False) and not FORCE_RETRAIN:
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded complete comparison checkpoint: {checkpoint_path}")
else:
    started = time.time()
    for epoch in range(1, EPOCHS + 1):
        train_metrics = classification_epoch(train_loader, training=True, max_batches=MAX_TRAIN_BATCHES)
        validation_metrics = classification_epoch(validation_loader, training=False, max_batches=MAX_EXPORT_BATCHES)
        scheduler.step()
        record = {"epoch": epoch, "train": train_metrics, "validation": validation_metrics}
        history.append(record)
        print(
            f"Epoch {epoch:02d}/{EPOCHS}: train={train_metrics['top1_accuracy']:.3%}, "
            f"validation={validation_metrics['top1_accuracy']:.3%}"
        )
        if validation_metrics["top1_accuracy"] > best_validation_accuracy:
            best_validation_accuracy = validation_metrics["top1_accuracy"]
            torch.save({
                "model_state_dict": model.state_dict(),
                "best_validation_accuracy": best_validation_accuracy,
                "history": history,
                "config": CONFIG,
                "class_names": train_aug_base.classes,
                "training_complete": False,
            }, checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    checkpoint["history"] = history
    checkpoint["training_complete"] = True
    checkpoint["completed_epochs"] = EPOCHS
    torch.save(checkpoint, checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Training completed in {(time.time() - started) / 60:.1f} minutes")

print(f"Best validation top-1: {best_validation_accuracy:.3%}")

## Export frozen representations before test evaluation

The SDM exemplar representation is the normalized, pooled 768-dimensional ConvNeXt vector immediately before the final linear classifier. Reference and calibration images use deterministic preprocessing.

In [ ]:
@torch.inference_mode()
def collect_outputs(loader, *, max_batches: int | None = None) -> dict[str, np.ndarray]:
    model.eval()
    collected = {key: [] for key in ("logits", "probabilities", "embeddings", "targets", "predictions")}
    for batch_number, (inputs, labels) in enumerate(loader):
        if max_batches is not None and batch_number >= max_batches:
            break
        inputs = inputs.to(DEVICE, non_blocking=True)
        features = model.features(inputs)
        embeddings = model.classifier[1](model.classifier[0](model.avgpool(features)))
        logits = model.classifier[2](embeddings)
        probabilities = torch.softmax(logits, dim=1)
        collected["logits"].append(logits.cpu())
        collected["probabilities"].append(probabilities.cpu())
        collected["embeddings"].append(embeddings.cpu())
        collected["targets"].append(labels.cpu())
        collected["predictions"].append(probabilities.argmax(dim=1).cpu())
    return {key: torch.cat(values).numpy() for key, values in collected.items()}


reference = collect_outputs(reference_loader, max_batches=MAX_EXPORT_BATCHES)
calibration_all = collect_outputs(calibration_loader, max_batches=MAX_EXPORT_BATCHES)

# Match by original dataset indices, so membership survives unequal class sizes.
standard_mask = np.isin(calibration_100_indices, calibration_25_indices)
calibration_sets = {
    "standard_25": {key: value[standard_mask] for key, value in calibration_all.items()},
    "enlarged_100": calibration_all,
}
np.savez_compressed(RUN_DIR / "reference_outputs.npz", **reference, dataset_indices=train_indices)
np.savez_compressed(RUN_DIR / "calibration_outputs.npz", **calibration_all, dataset_indices=calibration_100_indices)
print({name: len(arrays["targets"]) for name, arrays in calibration_sets.items()})

# SDM mapping, step by step

This is the frozen **identity-adaptor** variant: $h'=h$ is the ConvNeXt representation and $z'$ is its existing 47-class logit vector. It tests the activation and estimator without claiming to reproduce the paper's separately trained 1-D CNN adaptor.

Primary references:

- [SDM paper §3.1–3.2 and Appendix A.4–A.5](../third_party/sdm_activations/papers/2509.12760v5.pdf)
- [Upstream `sdm_model.py`](../third_party/sdm_activations/code/third_party_baselines/version_for_vbll_baselines/code/reexpress/sdm_model.py)
- [Upstream local-scaling tutorial](../third_party/sdm_activations/code/tutorials/local_scaling_tutorial_code.py)
- [Identity-adaptor ablation](../third_party/sdm_activations/documentation/scripts/sdm_activations_paper/aux_experiments/AdaptorRepresentationAblation.md)

Implementation pinned to upstream commit `06d295df31e0ab8ee13729244b9a7d4d5c4f91de`.

## Step 1 — Normalize the exemplar space

**Map:** paper §3.1.1 maps $h\rightarrow h'\rightarrow z'$. Here $h'=h$ and $z'$ remains the frozen ConvNeXt logit vector. Appendix A.4 and upstream `sdm_model.py` normalize with one global training-support mean and sample standard deviation.

**Why:** the distance geometry must be determined only by training support. Calibration estimates reliability in that fixed space; it does not refit the representation.

In [ ]:
reference_tensor = torch.as_tensor(reference["embeddings"], dtype=torch.float32)
embedding_mean = reference_tensor.mean()
embedding_std = reference_tensor.std(correction=1).clamp_min(torch.finfo(torch.float32).eps)


def normalize_embeddings(values: np.ndarray) -> np.ndarray:
    return ((torch.as_tensor(values, dtype=torch.float32) - embedding_mean) / embedding_std).numpy()


reference_embeddings = normalize_embeddings(reference["embeddings"])
calibration_embeddings = normalize_embeddings(calibration_all["embeddings"])
print({"global_mean": float(embedding_mean), "global_sample_std": float(embedding_std)})

## Step 2 — Retrieve exact nearest training exemplars

**Map:** paper §3.1.2 orders support examples by L2 distance. Upstream uses FAISS `IndexFlatL2`, which returns squared L2 distance. This batched PyTorch search returns the same exact ordering and distance definition.

**Why:** the neighbors supply the closest raw distance and the ordered label/prediction sequence used by Similarity $q$. One more neighbor than the largest observed true-class support is sufficient to observe the first mismatch.

In [ ]:
@torch.inference_mode()
def exact_squared_l2_neighbors(reference_values, query_values, *, neighbors, batch_size=192):
    support = torch.as_tensor(reference_values, dtype=torch.float32, device=DEVICE)
    support_norm = support.square().sum(dim=1)
    all_distances, all_indices = [], []
    neighbors = min(neighbors, len(support))
    for start in range(0, len(query_values), batch_size):
        query = torch.as_tensor(query_values[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        squared_l2 = (
            query.square().sum(dim=1, keepdim=True)
            + support_norm.unsqueeze(0)
            - 2 * query @ support.T
        ).clamp_min_(0)
        distances, indices = torch.topk(squared_l2, k=neighbors, dim=1, largest=False, sorted=True)
        all_distances.append(distances.cpu())
        all_indices.append(indices.cpu())
    return torch.cat(all_distances).numpy(), torch.cat(all_indices).numpy()


neighbors = min(len(reference["targets"]), int(np.bincount(reference["targets"], minlength=NUM_CLASSES).max()) + 1)
print({"calibration_queries": len(calibration_embeddings), "neighbors_per_query": neighbors})


def similarity_distance_signals(query_embeddings, query_predictions, batch_size=192):
    # Keep only q and the nearest exemplar. Full top-k groups are temporary;
    # support can contain thousands of examples/class in these datasets.
    nearest_distances, nearest_indices, similarities = [], [], []
    for start in range(0, len(query_embeddings), batch_size):
        stop = start + batch_size
        distances, indices = exact_squared_l2_neighbors(
            reference_embeddings, query_embeddings[start:stop], neighbors=neighbors, batch_size=batch_size
        )
        similarities.append(consecutive_correct_match_count(indices, query_predictions[start:stop]))
        nearest_distances.append(distances[:, :1])
        nearest_indices.append(indices[:, :1])
    return np.concatenate(nearest_distances), np.concatenate(nearest_indices), np.concatenate(similarities)


## Step 3 — Similarity $q$

**Map:** paper Eq. 4 counts consecutive nearest support examples that are both correctly classified and predicted as the query's predicted class. The count stops at the first failure, matching the cumulative-product construction in upstream `sdm_model.py`.

**Why:** $q$ measures local label/prediction consistency. A nearest mismatch gives $q=0$, which the SDM estimator treats as OOD-like.

In [ ]:
def consecutive_correct_match_count(neighbor_indices, query_predictions):
    neighbor_targets = reference["targets"][neighbor_indices]
    neighbor_predictions = reference["predictions"][neighbor_indices]
    matches = (
        (neighbor_targets == neighbor_predictions)
        & (neighbor_predictions == query_predictions[:, None])
    )
    return np.cumprod(matches.astype(np.int32), axis=1).sum(axis=1).astype(np.float32)


calibration_neighbor_distances, calibration_neighbor_indices, calibration_q_all = similarity_distance_signals(
    calibration_embeddings, calibration_all["predictions"]
)
print(pd.Series(calibration_q_all).describe(percentiles=[0.5, 0.9, 0.95]).rename("calibration q"))

## Step 4 — Distance $d$

**Map:** paper Eq. 5 builds a true-label-stratified empirical CDF of nearest distances for calibration examples with $q>0$. A query receives the minimum survival value $1-\mathrm{eCDF}_c$ across all 100 classes. Appendix A.4.2 specifies the exclusionary boundary implemented by `searchsorted(..., side='left')`.

**Why:** $d$ asks whether the query is no farther from support than examples under every classwise calibration distribution. The 25 and 100 tracks build separate CDFs, making effective calibration size the experimental variable.

In [ ]:
def build_class_distance_cdfs(distances, q, labels, number_of_classes=NUM_CLASSES):
    return [
        np.sort(distances[(labels == class_id) & (q > 0)])
        for class_id in range(number_of_classes)
    ]


def conservative_distance_quantile(distances, class_cdfs):
    per_class = np.zeros((len(distances), len(class_cdfs)), dtype=np.float32)
    for class_id, sorted_distances in enumerate(class_cdfs):
        if len(sorted_distances):
            ranks = np.searchsorted(sorted_distances, distances, side="left")
            per_class[:, class_id] = 1.0 - ranks / len(sorted_distances)
    return per_class.min(axis=1)

## Steps 5–7 — SDM activation, $q'$, and Algorithm 1

**Activation map:** paper Eq. 6 and Appendix A.4.1 give the stable identity

$$\operatorname{SDM}(z')=\operatorname{softmax}\left(z' d\log(2+q)\right).$$

Thus $q$ and $d$ form an example-specific inverse temperature. If $d=0$, the distribution is uniform; SDM otherwise preserves the raw-logit argmax.

**Estimator map:** Eq. 8 defines $q'=\min(q,(2+q)^{\operatorname{SDM}(z')_{\hat y}})$. Algorithm 1 scans calibration $q'$ cutoffs and accepts a High-Reliability region only when every true class's $(1-\alpha)$ probability quantile reaches $\alpha$. The current upstream source also requires a singleton thresholded prediction set.

**Why:** the activation changes confidence; Algorithm 1 converts that evidence into a selective prediction rule. Failure to find a finite region is a valid conservative result, not an implementation error.

In [ ]:
def sdm_activation(logits, q, d):
    logits_t = torch.as_tensor(logits, dtype=torch.float32)
    scale = torch.as_tensor(d).reshape(-1, 1) * torch.log(2.0 + torch.as_tensor(q).reshape(-1, 1))
    return torch.softmax(logits_t * scale, dim=1).numpy()


def rescaled_similarity(q, predicted_probability):
    return np.minimum(q, np.power(2.0 + q, predicted_probability))


def upstream_empirical_quantile(values, proportion):
    if len(values) == 0:
        return 0.0
    return float(np.sort(values)[min(int(proportion * len(values)), len(values) - 1)])


def fit_high_reliability_region(probabilities, q_prime, labels, *, alpha=TARGET_ALPHA):
    for candidate in np.unique(np.sort(q_prime[q_prime > 0])):
        retained = q_prime >= candidate
        thresholds = np.asarray([
            upstream_empirical_quantile(probabilities[retained & (labels == class_id), class_id], 1 - alpha)
            for class_id in range(NUM_CLASSES)
        ], dtype=np.float32)
        if np.all(thresholds >= alpha):
            return {"found": True, "q_prime_min": float(candidate), "thresholds": thresholds}
    return {"found": False, "q_prime_min": float("inf"), "thresholds": np.full(NUM_CLASSES, np.inf)}


def apply_high_reliability_region(probabilities, q_prime, predictions, region):
    if not region["found"]:
        return np.zeros(len(predictions), dtype=bool)
    above = probabilities >= region["thresholds"][None, :]
    return (
        (q_prime >= region["q_prime_min"])
        & (above.sum(axis=1) == 1)
        & above[np.arange(len(predictions)), predictions]
    )


# Equation-level regression checks.
test_logits = np.asarray([[1.0, 3.0]], dtype=np.float32)
np.testing.assert_allclose(
    sdm_activation(test_logits, np.asarray([np.e - 2]), np.asarray([1.0])),
    torch.softmax(torch.from_numpy(test_logits), dim=1).numpy(), rtol=1e-6,
)
np.testing.assert_allclose(
    sdm_activation(test_logits, np.asarray([100.0]), np.asarray([0.0])),
    np.asarray([[0.5, 0.5]]), rtol=1e-6,
)

## Fit and freeze both calibration tracks

Class cohorts are also frozen here. A class is `below_target` when its observed calibration accuracy is below 95%; otherwise it is `at_or_above_target`. These are development-set groupings, not population guarantees.

In [ ]:
def fit_track(name, arrays, full_mask):
    distances = calibration_neighbor_distances[full_mask, 0]
    q = calibration_q_all[full_mask]
    cdfs = build_class_distance_cdfs(distances, q, arrays["targets"])
    d = conservative_distance_quantile(distances, cdfs)
    probabilities = sdm_activation(arrays["logits"], q, d)
    predictions = arrays["predictions"]
    confidence = probabilities[np.arange(len(predictions)), predictions]
    q_prime = rescaled_similarity(q, confidence)
    region = fit_high_reliability_region(probabilities, q_prime, arrays["targets"])

    totals = np.bincount(arrays["targets"], minlength=NUM_CLASSES)
    correct = np.bincount(
        arrays["targets"][arrays["targets"] == predictions], minlength=NUM_CLASSES
    )
    accuracy = np.divide(correct, totals, out=np.full(NUM_CLASSES, np.nan), where=totals > 0)
    cohorts = np.where(accuracy >= TARGET_ALPHA, "at_or_above_target", "below_target")
    cohorts[totals == 0] = "unobserved"
    return {
        "name": name, "cdfs": cdfs, "d": d, "q": q,
        "probabilities": probabilities, "q_prime": q_prime,
        "region": region, "class_accuracy": accuracy, "class_cohorts": cohorts,
    }


full_masks = {
    "standard_25": standard_mask,
    "enlarged_100": np.ones(len(calibration_all["targets"]), dtype=bool),
}
tracks = {
    name: fit_track(name, calibration_sets[name], full_masks[name])
    for name in ("standard_25", "enlarged_100")
}

fit_summary = pd.DataFrame([{
    "track": name,
    "calibration_examples": len(calibration_sets[name]["targets"]),
    "smallest_class_cdf": min(map(len, track["cdfs"])),
    "d_zero_fraction": float((track["d"] == 0).mean()),
    "mean_d": float(track["d"].mean()),
    "finite_hr_region": track["region"]["found"],
    "q_prime_min": track["region"]["q_prime_min"] if track["region"]["found"] else np.nan,
    "below_target_classes": int((track["class_cohorts"] == "below_target").sum()),
} for name, track in tracks.items()])
display(fit_summary)

## Fit and freeze temperature-scaled softmax baselines

Following [Guo et al. (2017)](https://proceedings.mlr.press/v70/guo17a.html), fit one positive scalar $T$ using calibration negative log-likelihood (NLL), then evaluate $\operatorname{softmax}(z/T)$ with that frozen value. Fit separately on the nested 25/class and 100/class calibration sets. Model weights, raw-logit predictions, SDM fits, and test labels do not enter this optimization.

The deterministic solver minimizes convex NLL in inverse temperature $1/T$ by derivative bisection, with $T$ restricted to `[0.01, 100]` for numerical stability. It compares the candidate with unscaled `T=1` and both bounds. The table reports calibration NLL before/after and whether a search bound was reached. A bound fit indicates the finite search range limited the solution. In smoke mode, the small partial calibration slice makes these diagnostics pipeline checks only.


In [ ]:
def temperature_probabilities(logits, temperature):
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be finite and positive.")
    scaled = np.asarray(logits, dtype=np.float64) / temperature
    scaled -= scaled.max(axis=1, keepdims=True)
    exponential = np.exp(scaled)
    return exponential / exponential.sum(axis=1, keepdims=True)


def fit_temperature(logits, labels):
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    if not len(labels) or not np.isfinite(logits).all():
        raise ValueError("Temperature fitting requires nonempty finite calibration logits.")
    centered = logits - logits.max(axis=1, keepdims=True)
    true_logits = centered[np.arange(len(labels)), labels]

    def nll(inverse_temperature):
        scaled = centered * inverse_temperature
        return float((np.log(np.exp(scaled).sum(axis=1)) - true_logits * inverse_temperature).mean())

    def derivative(inverse_temperature):
        probabilities = temperature_probabilities(centered, 1 / inverse_temperature)
        return float(((probabilities * centered).sum(axis=1) - true_logits).mean())

    lower, upper = 0.01, 100.0
    for _ in range(60):
        midpoint = (lower + upper) / 2
        if derivative(midpoint) > 0:
            upper = midpoint
        else:
            lower = midpoint
    candidates = [1.0, 0.01, 100.0, (lower + upper) / 2]
    best = min(candidates, key=nll)
    return {
        "temperature": float(1 / best),
        "calibration_examples": len(labels),
        "calibration_nll_before": nll(1.0),
        "calibration_nll_after": nll(best),
        "at_search_bound": bool(np.isclose(best, 0.01) or np.isclose(best, 100.0)),
    }


for name, track in tracks.items():
    arrays = calibration_sets[name]
    track["temperature_fit"] = fit_temperature(arrays["logits"], arrays["targets"])
temperature_table = pd.DataFrame([
    {"track": name, **track["temperature_fit"]} for name, track in tracks.items()
])
display(temperature_table)


# Freeze complete: load the official EMNIST Balanced test set

Everything above—model weights, both CDF collections, HR thresholds, cohort maps, and both softmax temperatures—is now fixed. The test labels below are used only to calculate final metrics and never to alter either method.

In [ ]:
# First access to the official test split: all model/calibration choices are frozen.
test_base = load_dataset(SPEC, DATA_DIR / "datasets", train=False, transform=eval_transform, download=DOWNLOAD)
test_indices = smoke_test_indices(test_base.targets, NUM_CLASSES) if SMOKE else np.arange(len(test_base))
test_loader = DataLoader(Subset(test_base, test_indices), shuffle=False, **loader_options)
test = collect_outputs(test_loader, max_batches=MAX_EXPORT_BATCHES)
np.savez_compressed(RUN_DIR / "test_outputs.npz", **test, dataset_indices=test_indices)
test_embeddings = normalize_embeddings(test["embeddings"])
test_neighbor_distances, test_neighbor_indices, test_q = similarity_distance_signals(
    test_embeddings, test["predictions"]
)
test_d_nearest = test_neighbor_distances[:, 0]
print({"official_test_examples": len(test["targets"]), "loaded_after_freeze": True, "smoke": SMOKE})


## Apply both frozen SDM tracks to the same test images

Only $d$ differs between the tracks because its classwise CDF is calibrated from 25 versus 100 examples/class. Test $q$, raw logits, predictions, and nearest-support distances are identical.

In [ ]:
for track in tracks.values():
    track["test_temperature_probabilities"] = temperature_probabilities(test["logits"], track["temperature_fit"]["temperature"])
    track["test_d"] = conservative_distance_quantile(test_d_nearest, track["cdfs"])
    track["test_probabilities"] = sdm_activation(test["logits"], test_q, track["test_d"])
    track["test_confidence"] = track["test_probabilities"][
        np.arange(len(test["predictions"])), test["predictions"]
    ]
    track["test_q_prime"] = rescaled_similarity(test_q, track["test_confidence"])
    track["test_hr_admitted"] = apply_high_reliability_region(
        track["test_probabilities"], track["test_q_prime"], test["predictions"], track["region"]
    )

# Better metrics than a JSON artifact

- **Macro/micro precision and recall:** classification performance; macro averages observed true classes equally.
- **NLL, Brier, ECE:** probability quality and calibration.
- **Error AUROC/AUPRC:** how well low confidence identifies wrong predictions.
- **Risk–coverage and AURC:** error rate as increasingly uncertain examples are rejected; lower AURC is better.
- **Coverage at 95% accuracy:** largest prefix whose empirical selective accuracy reaches the target.
- **Accuracy at fixed coverage:** direct operating-point comparisons.
- **HR coverage and accuracy:** the upstream Algorithm 1 selective region, reported even when coverage is zero.

The individual CSV remains important for auditing, but these aggregate metrics answer whether confident errors are actually reduced.

In [ ]:
def classification_metrics(labels, predictions, number_of_classes, admitted=None):
    labels, predictions = np.asarray(labels), np.asarray(predictions)
    admitted = np.ones(len(labels), dtype=bool) if admitted is None else np.asarray(admitted, dtype=bool)
    support = np.bincount(labels, minlength=number_of_classes)
    predicted = np.bincount(predictions[admitted], minlength=number_of_classes)
    correct = admitted & (predictions == labels)
    tp = np.bincount(labels[correct], minlength=number_of_classes)
    precision = np.divide(tp, predicted, out=np.zeros(number_of_classes, dtype=float), where=predicted > 0)
    recall = np.divide(tp, support, out=np.full(number_of_classes, np.nan), where=support > 0)
    observed = support > 0
    per_class = pd.DataFrame({
        "class_id": np.arange(number_of_classes), "support": support,
        "admitted_predictions": predicted, "true_positives": tp,
        "precision": precision, "recall": recall,
    })
    summary = {
        "accuracy": float((predictions == labels).mean()) if len(labels) else np.nan,
        "coverage": float(admitted.mean()) if len(labels) else 0.0,
        "macro_precision": float(precision[observed].mean()) if observed.any() else np.nan,
        "macro_recall": float(recall[observed].mean()) if observed.any() else np.nan,
        "micro_precision": float(tp.sum() / admitted.sum()) if admitted.any() else np.nan,
        "micro_recall": float(tp.sum() / len(labels)) if len(labels) else np.nan,
    }
    return per_class, summary


In [ ]:
def binary_ranking_metrics(labels, scores):
    labels = np.asarray(labels, dtype=bool)
    order = np.argsort(-np.asarray(scores), kind="stable")
    y = labels[order].astype(float)
    positives, negatives = y.sum(), len(y) - y.sum()
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    if positives and negatives:
        tpr = np.r_[0.0, tp / positives, 1.0]
        fpr = np.r_[0.0, fp / negatives, 1.0]
        auroc = float(np.trapezoid(tpr, fpr))
    else:
        auroc = np.nan
    precision = tp / np.arange(1, len(y) + 1)
    recall = tp / positives if positives else np.full(len(y), np.nan)
    auprc = float(np.sum((recall - np.r_[0.0, recall[:-1]]) * precision)) if positives else np.nan
    return auroc, auprc


def probability_metrics(probabilities, labels, predictions, bins=15):
    n = len(labels)
    confidence = probabilities[np.arange(n), predictions]
    correct = predictions == labels
    nll = -np.log(np.clip(probabilities[np.arange(n), labels], 1e-12, 1)).mean()
    one_hot = np.eye(probabilities.shape[1], dtype=np.float32)[labels]
    brier = np.square(probabilities - one_hot).sum(axis=1).mean()
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (confidence > low) & (confidence <= high)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    error_auroc, error_auprc = binary_ranking_metrics(~correct, 1 - confidence)
    order = np.argsort(-confidence, kind="stable")
    cumulative_accuracy = np.cumsum(correct[order]) / np.arange(1, n + 1)
    risk = 1 - cumulative_accuracy
    coverage = np.arange(1, n + 1) / n
    aurc = float(np.trapezoid(risk, coverage))
    eligible = np.flatnonzero(cumulative_accuracy >= TARGET_ALPHA)
    coverage_at_alpha = float((eligible[-1] + 1) / n) if len(eligible) else 0.0
    fixed = {}
    for target_coverage in (0.10, 0.25, 0.50, 0.75, 1.00):
        count = max(1, int(np.floor(n * target_coverage)))
        fixed[f"accuracy_at_{int(target_coverage * 100)}pct_coverage"] = float(correct[order[:count]].mean())
    return {
        **classification_metrics(labels, predictions, probabilities.shape[1])[1],
        "nll": float(nll), "brier": float(brier),
        "ece_15": float(ece), "error_auroc": error_auroc, "error_auprc": error_auprc,
        "aurc": aurc, "coverage_at_95pct_accuracy": coverage_at_alpha,
        **fixed, "curve_coverage": coverage, "curve_risk": risk,
    }


methods = {"softmax": test["probabilities"]}
methods.update({f"temperature_{name}": track["test_temperature_probabilities"] for name, track in tracks.items()})
methods.update({f"sdm_{name}": track["test_probabilities"] for name, track in tracks.items()})
metric_records, curves = [], {}
for method, probabilities in methods.items():
    metrics = probability_metrics(probabilities, test["targets"], test["predictions"])
    curves[method] = (metrics.pop("curve_coverage"), metrics.pop("curve_risk"))
    metric_records.append({"method": method, **metrics})

metrics_table = pd.DataFrame(metric_records).set_index("method")
display(metrics_table.style.format("{:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for method, (coverage, risk) in curves.items():
    axes[0].plot(coverage, risk, label=method)
axes[0].set(title="Test risk–coverage curves", xlabel="Coverage", ylabel="Selective risk", xlim=(0, 1))
axes[0].grid(alpha=0.25)
axes[0].legend()

width = 0.35
x = np.arange(len(tracks))
hr_coverage = [tracks[name]["test_hr_admitted"].mean() for name in tracks]
hr_accuracy = []
for name in tracks:
    admitted = tracks[name]["test_hr_admitted"]
    hr_accuracy.append(
        (test["predictions"][admitted] == test["targets"][admitted]).mean() if admitted.any() else np.nan
    )
axes[1].bar(x - width / 2, hr_coverage, width, label="coverage")
axes[1].bar(x + width / 2, np.nan_to_num(hr_accuracy), width, label="accuracy")
axes[1].set(title="Algorithm 1 HR test result", ylabel="Proportion", xticks=x, xticklabels=list(tracks), ylim=(0, 1))
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend()
plt.tight_layout()
fig.savefig(RUN_DIR / "risk_coverage_and_hr.png", dpi=160)
plt.show()

hr_table = pd.DataFrame({
    "track": list(tracks),
    "region_found_on_calibration": [tracks[name]["region"]["found"] for name in tracks],
    "test_admitted": [int(tracks[name]["test_hr_admitted"].sum()) for name in tracks],
    "test_coverage": hr_coverage,
    "test_accuracy_when_admitted": hr_accuracy,
})
display(hr_table)

## Low- versus high-performing class cohorts on held-out test data

Results are shown two ways. `true_class` is retrospective error analysis using the known label. `predicted_class` is deployable grouping because it uses only the model's prediction. Each track uses the cohort map frozen from its own calibration set.

In [ ]:
cohort_rows = []
correct = test["predictions"] == test["targets"]
for name, track in tracks.items():
    for assignment, class_ids in {
        "true_class": test["targets"],
        "predicted_class": test["predictions"],
    }.items():
        example_cohorts = track["class_cohorts"][class_ids]
        for cohort in ("below_target", "at_or_above_target"):
            mask = example_cohorts == cohort
            if mask.any():
                admitted = track["test_hr_admitted"] & mask
                cohort_rows.append({
                    "track": name, "assignment": assignment, "cohort": cohort,
                    "examples": int(mask.sum()), "accuracy": float(correct[mask].mean()),
                    "softmax_confidence": float(test["probabilities"].max(axis=1)[mask].mean()),
                    "sdm_confidence": float(track["test_confidence"][mask].mean()),
                    "mean_q": float(test_q[mask].mean()), "mean_d": float(track["test_d"][mask].mean()),
                    "hr_coverage": float(admitted.sum() / mask.sum()),
                    "hr_accuracy": float(correct[admitted].mean()) if admitted.any() else np.nan,
                })
cohort_table = pd.DataFrame(cohort_rows)
display(cohort_table)

## Which classes are confused?

Rows are the **true class** and columns are the **predicted class**. The first matrix shows the fraction of each true class assigned to each prediction (the diagonal is class recall). The second view hides correct predictions and zooms into the 20 classes most involved in errors; its cells show error counts. The table lists the most frequent directed confusions across all classes.

These plots use the existing held-out predictions. Raw softmax, temperature-scaled softmax, and both SDM tracks share those predictions, so one classification confusion matrix applies to all five methods before abstention. In smoke mode, unobserved classes have zero rows and the plots are only pipeline checks.

In [ ]:
confusion_class_names = np.asarray(test_base.classes)
confusion_class_count = len(confusion_class_names)
confusion_counts = np.zeros((confusion_class_count, confusion_class_count), dtype=np.int64)
np.add.at(confusion_counts, (test["targets"], test["predictions"]), 1)
confusion_support = confusion_counts.sum(axis=1)
confusion_rates = np.divide(
    confusion_counts,
    confusion_support[:, None],
    out=np.zeros_like(confusion_counts, dtype=float),
    where=confusion_support[:, None] > 0,
)
plot_suffix = "smoke" if SMOKE else "full"

fig, ax = plt.subplots(figsize=(18, 16), layout="constrained")
image = ax.imshow(confusion_rates, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
ax.set(
    title=f"Held-out EMNIST Balanced confusion matrix ({plot_suffix})",
    xlabel="Predicted class", ylabel="True class",
    xticks=np.arange(confusion_class_count), yticks=np.arange(confusion_class_count),
)
ax.set_xticklabels(confusion_class_names, rotation=90, fontsize=6)
ax.set_yticklabels(confusion_class_names, fontsize=6)
fig.colorbar(image, ax=ax, shrink=0.75, label="Fraction of true-class images")
fig.savefig(RUN_DIR / f"confusion_matrix_{plot_suffix}.png", dpi=160)
plt.show()

confusion_errors = confusion_counts.copy()
np.fill_diagonal(confusion_errors, 0)
error_involvement = confusion_errors.sum(axis=0) + confusion_errors.sum(axis=1)
focus_classes = np.flatnonzero(error_involvement > 0)
focus_classes = focus_classes[np.argsort(-error_involvement[focus_classes], kind="stable")[:20]]
focus_classes = np.sort(focus_classes)
if len(focus_classes):
    focused_errors = confusion_errors[np.ix_(focus_classes, focus_classes)]
    fig, ax = plt.subplots(figsize=(12, 10), layout="constrained")
    image = ax.imshow(focused_errors, cmap="Oranges", vmin=0, interpolation="nearest")
    ax.set(
        title=f"Errors among the {len(focus_classes)} classes most involved in confusion",
        xlabel="Predicted class", ylabel="True class",
        xticks=np.arange(len(focus_classes)), yticks=np.arange(len(focus_classes)),
    )
    ax.set_xticklabels(confusion_class_names[focus_classes], rotation=90, fontsize=9)
    ax.set_yticklabels(confusion_class_names[focus_classes], fontsize=9)
    for row, column in np.argwhere(focused_errors > 0):
        count = focused_errors[row, column]
        ax.text(column, row, str(count), ha="center", va="center", fontsize=8,
                color="white" if count > focused_errors.max() / 2 else "black")
    fig.colorbar(image, ax=ax, shrink=0.8, label="Misclassified images (correct diagonal hidden)")
    fig.savefig(RUN_DIR / f"confusion_errors_{plot_suffix}.png", dpi=160)
    plt.show()
else:
    print("No misclassifications in this test slice.")

confused_true, confused_predicted = np.nonzero(confusion_errors)
confusion_pairs = pd.DataFrame({
    "true_class": confusion_class_names[confused_true],
    "predicted_as": confusion_class_names[confused_predicted],
    "errors": confusion_errors[confused_true, confused_predicted],
    "fraction_of_true_class": confusion_rates[confused_true, confused_predicted],
}).sort_values("errors", ascending=False, kind="stable").head(20)
display(confusion_pairs.reset_index(drop=True).style.format({"fraction_of_true_class": "{:.1%}"}))


## Precision and recall by class and after abstention

For each class, **precision** is correct predictions divided by predictions of that class; **recall** is correct predictions divided by all images truly in that class. **Micro** precision/recall pool counts across classes; at full coverage both equal top-1 accuracy. **Macro** precision/recall average over classes present in the true test labels. Precision is set to zero when a class has no admitted predictions. Recall for an unobserved true class is undefined (`NaN`), and that class is excluded from macro averages. The per-class table retains every EMNIST Balanced class and its support, so smoke-mode omissions remain visible.

For the HR rows, abstained images remain in recall's denominator and do not enter precision's denominator. Thus micro precision equals selective accuracy and micro recall equals coverage × selective accuracy. With no admissions, micro precision is undefined and recall is zero. Full-coverage class metrics apply to all five methods because they share the raw-logit predictions.


In [ ]:
class_metrics_table, _ = classification_metrics(
    test["targets"], test["predictions"], len(test_base.classes)
)
class_metrics_table.insert(1, "class_name", test_base.classes)
display(class_metrics_table.style.format({"precision": "{:.4f}", "recall": "{:.4f}"}))

selective_rows = []
for name, track in tracks.items():
    _, summary = classification_metrics(
        test["targets"], test["predictions"], len(test_base.classes),
        admitted=track["test_hr_admitted"],
    )
    summary.pop("accuracy")  # Full-coverage accuracy is already in metrics_table.
    selective_rows.append({"method": f"sdm_{name}", "region": "HR", **summary})
selective_metrics_table = pd.DataFrame(selective_rows)
display(selective_metrics_table)


## Recall versus coverage

At a confidence threshold, **coverage** is the fraction of test images admitted. **Macro recall** is the average, across observed true classes, of correctly classified admitted images divided by all test images in that class. An abstention counts as a missed true-class prediction. At full coverage this is the classifier's usual macro recall; for balanced test classes it equals top-1 accuracy. For imbalanced test data, macro recall can differ from accuracy.

Each curve uses the raw softmax, temperature-scaled softmax, or SDM confidence for the shared predicted class. Equal-confidence images enter together, so every plotted marker is attainable by a single confidence threshold. Connecting segments guide the eye and do not imply that a tied group can be split. This is retrospective test analysis; the curves do not fit or change any admission threshold. Smoke mode averages only classes present in its partial slice.

In [ ]:
def macro_recall_coverage(confidence, labels, predictions, number_of_classes):
    confidence = np.asarray(confidence)
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    class_support = np.bincount(labels, minlength=number_of_classes)
    observed_classes = np.count_nonzero(class_support)
    if not len(labels) or not observed_classes:
        return np.asarray([0.0]), np.asarray([0.0])
    order = np.argsort(-confidence, kind="stable")
    sorted_confidence = confidence[order]
    # Only evaluate after a complete group of equal-confidence images.
    group_ends = np.r_[np.flatnonzero(sorted_confidence[:-1] != sorted_confidence[1:]), len(order) - 1]
    correct_recall_contribution = (
        (predictions[order] == labels[order]) / class_support[labels[order]] / observed_classes
    )
    cumulative_recall = np.cumsum(correct_recall_contribution)
    return (
        np.r_[0.0, (group_ends + 1) / len(order)],
        np.r_[0.0, cumulative_recall[group_ends]],
    )


recall_confidences = {
    method: probabilities[np.arange(len(test["predictions"])), test["predictions"]]
    for method, probabilities in methods.items()
}
fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")
recall_curves = {}
for method, confidence in recall_confidences.items():
    coverage, recall = macro_recall_coverage(
        confidence, test["targets"], test["predictions"], len(test_base.classes)
    )
    recall_curves[method] = (coverage, recall)
    ax.plot(coverage, recall, marker=".", markersize=2, label=method)
ax.set(
    title=f"Held-out macro recall versus coverage ({plot_suffix})",
    xlabel="Coverage (fraction of test images admitted)",
    ylabel="Macro recall (abstentions count as misses)",
    xlim=(0, 1), ylim=(0, 1),
)
ax.grid(alpha=0.25)
ax.legend()
fig.savefig(RUN_DIR / f"recall_coverage_{plot_suffix}.png", dpi=160)
recall_curve_table = pd.concat([
    pd.DataFrame({"method": method, "coverage": coverage, "macro_recall": recall})
    for method, (coverage, recall) in recall_curves.items()
], ignore_index=True)
plt.show()


## Save auditable individual results and compact metric tables

The CSV has one row per test image, so each confident error and nearest exemplar can be inspected. JSON is retained only for configuration and provenance; it is not the evaluation dataset.

In [ ]:
suffix = "smoke" if SMOKE else "full"
individual = pd.DataFrame({
    "test_index": test_indices,
    "true_class_id": test["targets"],
    "true_class_name": np.asarray(test_base.classes)[test["targets"]],
    "predicted_class_id": test["predictions"],
    "predicted_class_name": np.asarray(test_base.classes)[test["predictions"]],
    "correct": correct,
    "softmax_confidence": test["probabilities"].max(axis=1),
    "q": test_q,
    "nearest_distance_squared": test_d_nearest,
    "nearest_reference_dataset_index": train_indices[test_neighbor_indices[:, 0]],
})
for name, track in tracks.items():
    individual[f"{name}_temperature_confidence"] = track["test_temperature_probabilities"].max(axis=1)
    individual[f"{name}_d"] = track["test_d"]
    individual[f"{name}_sdm_confidence"] = track["test_confidence"]
    individual[f"{name}_q_prime"] = track["test_q_prime"]
    individual[f"{name}_hr_admitted"] = track["test_hr_admitted"]
    individual[f"{name}_true_cohort"] = track["class_cohorts"][test["targets"]]
    individual[f"{name}_predicted_cohort"] = track["class_cohorts"][test["predictions"]]

individual_path = RUN_DIR / f"test_individual_results_{suffix}.csv.gz"
metrics_path = RUN_DIR / f"test_reliability_metrics_{suffix}.csv"
cohort_path = RUN_DIR / f"test_cohort_metrics_{suffix}.csv"
hr_path = RUN_DIR / f"test_hr_metrics_{suffix}.csv"
class_metrics_path = RUN_DIR / f"test_class_precision_recall_{suffix}.csv"
selective_metrics_path = RUN_DIR / f"test_selective_precision_recall_{suffix}.csv"
temperature_path = RUN_DIR / f"temperature_fits_{suffix}.csv"
recall_path = RUN_DIR / f"test_recall_coverage_{suffix}.csv"
metadata_path = RUN_DIR / f"experiment_metadata_{suffix}.json"
individual.to_csv(individual_path, index=False, compression="gzip")
metrics_table.to_csv(metrics_path)
cohort_table.to_csv(cohort_path, index=False)
hr_table.to_csv(hr_path, index=False)
class_metrics_table.to_csv(class_metrics_path, index=False)
selective_metrics_table.to_csv(selective_metrics_path, index=False)
temperature_table.to_csv(temperature_path, index=False)
recall_curve_table.to_csv(recall_path, index=False)

metadata = {
    "config": CONFIG,
    "dataset_source": SPEC.source_url,
    "class_names": CLASS_NAMES,
    "status": "pipeline_check" if SMOKE else "complete_benchmark",
    "split_manifest": str(RUN_DIR / "split_indices.npz"),
    "upstream_sdm_commit": "06d295df31e0ab8ee13729244b9a7d4d5c4f91de",
    "variant": "frozen_identity_adaptor",
    "split_sizes": {"train_reference": len(train_indices), "validation": len(validation_indices), "standard_calibration": len(calibration_25_indices), "enlarged_calibration": len(calibration_100_indices), "test": len(test_indices)},
    "best_validation_accuracy": best_validation_accuracy,
    "temperature_fits": json.loads(temperature_table.to_json(orient="records")),
    "classification_conventions": {"macro_classes": "observed true classes", "zero_predicted_precision": 0.0, "abstentions": "false negatives for recall"},
    "fit_summary": json.loads(fit_summary.to_json(orient="records")),
    "artifacts": {"individual_results": str(individual_path), "metrics": str(metrics_path), "cohorts": str(cohort_path), "hr": str(hr_path), "class_precision_recall": str(class_metrics_path), "selective_precision_recall": str(selective_metrics_path), "temperature_fits": str(temperature_path), "recall_coverage": str(recall_path)},
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

print("Saved:")
for path in (individual_path, metrics_path, cohort_path, hr_path, class_metrics_path, selective_metrics_path, temperature_path, recall_path, metadata_path):
    print(f"  {path}")

## How to read the comparison

- Prefer the enlarged track if it improves AURC, error-detection AUROC/AUPRC, and coverage at 95% accuracy without degrading probability metrics.
- HR accuracy is meaningless without HR coverage; zero coverage is an honest outcome when no finite 47-class region exists.
- Full-coverage top-1 accuracy, precision, and recall are identical across raw softmax, positive-temperature softmax, and SDM because this activation preserves the raw-logit argmax. The scientific question is whether SDM confidence ranks errors and supports abstention better.
- Read the fitted-temperature table alongside NLL/ECE. Temperature changes probability calibration and may change confidence ordering between images, while preserving each image's raw-logit prediction. A search-bound fit is reported explicitly.
- A better enlarged result supports the hypothesis that 25 calibration examples/class were too sparse. A similar or worse result points toward representation/adaptor limitations rather than calibration size alone.
- Test results are final evaluation evidence. Any subsequent change to alpha, split size, adaptor, or thresholds defines a new experiment and requires a fresh held-out evaluation protocol.